<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/simple_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone 'https://github.com/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/'

In [ ]:
!pip install dspy

In [ ]:
import dspy

In [ ]:
lm = dspy.LM('ollama_chat/qwen3:8b', api_base = 'http://localhost:11434', api_key='', max_tokens=2048)
dspy.configure(lm=lm)

In [ ]:
import pandas as pd

instructions_df = pd.read_excel('instructions.xlsx')

In [ ]:
instructions_df

In [ ]:
prompts_df = instructions_df.drop_duplicates(subset=[0], inplace=False)

In [ ]:
prompts_df

In [ ]:
prompts_df = prompts_df.drop(columns=['Unnamed: 0'], inplace=False)

In [ ]:
prompts_df

In [ ]:
prompts = []

for classes, instructions in prompts_df.values:
    examples = dspy.Example(classes=classes, instructions=instructions).with_inputs("classes", "instructions")
    prompts.append(examples)

In [ ]:
prompts

In [ ]:
import pandas as pd

class PromptTool:
  """Tool for retrieving appropriate instruction from excel spreadsheet."""
  def __init__(self):
    self.prompts = prompts

  def search_prompts(self, query: str, user_id: str = "default_user", limit: int=4) -> str:
    """Search for the relevant instruction prompt given a user's input"""
    try:
      prompt = self.prompts.search(query, user_id=user_id, limit=limit)
      if not prompt:
        return "No relevant prompt found"

      prompt_text = "Using the following promt:\n"
      for i, prompt in enumerate(prompts["instructions"]):
        prompt_text += f"{i}.{prompt['instructions']}\n"
        return prompt_text
    except Exception as e:
      return f"Error searching prompts: {str(e)}"

  def update_prompt(self, prompt_class: str, new_prompt: str) -> str:
    """Update the existing instruction prompt if appropriate to do so."""
    try:
      self.prompt.update(prompt_class, new_prompt)
      return f"Updated prompt with new instructions: {new_prompt}"
    except Exception as e:
      return f"Error updating instructions: {str(e)}"




In [ ]:
class ChatbotSignature(dspy.Signature):
  """
  You are a Dungeons & Dragons player agent and have access to a spreadsheet of
  instruction prompts paired with a category of player behaviour. Whenever you
  answer a user's input, respond according to the instructions which best match
  the situation the user's input creates.
  """
  user_input: str = dspy.InputField()
  response: str = dspy.OutputField()

class InstructedChatbot(dspy.Module):
